# Backends and speed

The anchor is fitted by L-BFGS. Four implementations are available and
`backend="auto"` chooses between two of them on a measured threshold.

In [1]:
from lairnet import available_backends

available_backends()

('numpy', 'numba', 'torch', 'sklearn')

## What `auto` does

NumPy below 5000 samples, numba at or above when it is importable. torch is
never chosen automatically.

In [2]:
from lairnet import AnchorNet

probe = AnchorNet(n_hidden=10)
for n in (500, 4999, 5000, 50_000):
    print(f"n={n:<7} -> {probe._resolve_backend(n)}")

n=500     -> numpy
n=4999    -> numpy
n=5000    -> numba
n=50000   -> numba


The threshold is the smallest size where numba was **measured** to win in the
regime the package actually runs, not an interpolated crossover — see
`benchmarks/RESULTS.md`.

torch is faster still at larger sample sizes and deliberately not automatic: it
is a multi-gigabyte dependency, and a default that changes with whatever else is
installed is not a default. Ask for it explicitly if you want it.

```{warning}
Do not reach for `device="cuda"` on problems of this shape. CUDA was slower than
the CPU in every regime measured, because transfers dominate an optimisation
this small.
```

In [3]:
import time

from sklearn.datasets import make_friedman1

X, y = make_friedman1(n_samples=1500, random_state=0)

for backend in ("numpy", "sklearn"):
    start = time.perf_counter()
    fit = AnchorNet(n_hidden=50, backend=backend, max_iter=400,
                    random_state=0).fit(X, y)
    print(f"{backend:<9} {time.perf_counter() - start:6.2f}s   "
          f"objective {fit.loss_:.6g}   converged {fit.converged_}")

C:\Users\yuvra\AppData\Local\Temp\ipykernel_32832\2418099634.py:10: ConvergenceWarning: AnchorNet stopped at max_iter=400 without converging (loss=0.00336923). The anchor is whatever L-BFGS had reached, so the iteration cap is acting as an implicit regulariser. Raise max_iter, loosen tol, or set alpha deliberately.
  random_state=0).fit(X, y)


numpy       1.59s   objective 0.00336923   converged False


sklearn     1.43s   objective 0.00228968   converged False


C:\Users\yuvra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


## The anchor on its own

`AnchorNet` is a scikit-learn transformer, so `fit_transform` gives you the
target-aware representation directly. It is the quickest way to see whether
LAIR-Net has anything to align to on your problem.

In [4]:
anchor = AnchorNet(n_hidden=20, max_iter=400, random_state=0)
anchor.fit_transform(X, y).shape

C:\Users\yuvra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:919: ConvergenceWarning: AnchorNet stopped at max_iter=400 without converging (loss=0.00785528). The anchor is whatever L-BFGS had reached, so the iteration cap is acting as an implicit regulariser. Raise max_iter, loosen tol, or set alpha deliberately.
  return self.fit(X, y, **fit_params).transform(X)


(1500, 20)

In [5]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

make_pipeline(
    AnchorNet(n_hidden=20, max_iter=400, random_state=0), Ridge()
).fit(X, y).score(X, y)

0.9991381267971985